In [0]:
%sql

CREATE OR REPLACE TABLE fraud_mlops.gold.production_ground_truth
USING DELTA
AS

SELECT
    transaction_id,
    CAST(NULL AS BIGINT) AS is_fraud,
    'PENDING' AS label_status,
    CAST(NULL AS TIMESTAMP) AS label_available_timestamp

FROM fraud_mlops.gold.production_drift_window;

In [0]:
%sql

SELECT
    label_status,
    COUNT(*) AS rows,
    COUNT(is_fraud) AS labels_available

FROM fraud_mlops.gold.production_ground_truth

GROUP BY label_status;

In [0]:
%sql

MERGE INTO fraud_mlops.gold.production_ground_truth AS target

USING fraud_mlops.gold.production_drift_window AS source

ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN
UPDATE SET
    target.is_fraud = source.is_fraud,
    target.label_status = 'AVAILABLE',
    target.label_available_timestamp = current_timestamp();

In [0]:
%sql

SELECT
    label_status,
    COUNT(*) AS rows,
    SUM(is_fraud) AS actual_fraud_rows

FROM fraud_mlops.gold.production_ground_truth

GROUP BY label_status;

In [0]:
%sql

SELECT
    p.transaction_id,
    p.fraud_probability,
    p.fraud_prediction,
    g.is_fraud,
    g.label_status

FROM fraud_mlops.gold.production_predictions p

INNER JOIN fraud_mlops.gold.production_ground_truth g
    ON p.transaction_id = g.transaction_id

LIMIT 10;